# E4 · Ablación de inferencia sin reentrenamiento

**Objetivo.** Medir si el flip horizontal y la agregación por identidad reducen ambigüedades de E4 sin modificar sus pesos.  
**Pregunta.** ¿`horizontal-flip TTA` y `top-2 mean` mejoran Rank-1 o mAP en validación frente a `single + max`?  
**Hipótesis.** El pooling esférico atenúa variaciones de pose y `top-2 mean` evita que una sola foto impostora domine el score.  
**Intervención.** Seis combinaciones: representación original/TTA × `max`/`top-2 mean`/prototipo esférico.  
**Control.** E4 semilla 46, versión registrada 19, sin cambios de detector, encoder, MLP, fotos o splits.  
**Muestra y selección.** Solo `validation` de los datasets refinados v2. Rank-1 macro por especie es la métrica primaria; mAP macro desempata.  
**Límite previo.** Este diagnóstico no carga benchmarks de evaluación, no reentrena y no acredita mejoras en test.


In [ ]:
import sys, subprocess

if sys.platform != "linux" or not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Use Colab with Python 3.11–3.13")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "pettrace-ml[notebooks]==0.2.28"])


## Fase 1. Preparación y preprocesamiento de datos

La release pública v2 aporta fotos, splits y cajas YOLO inmutables. La carga verifica el corpus completo, pero esta ablación conserva exclusivamente filas `val`; no calcula embeddings ni métricas sobre `test`.


In [ ]:
import json, os, uuid
from datetime import datetime, timezone
from pathlib import Path

import mlflow
from google.colab import userdata

for name in ("HF_TOKEN", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    value = userdata.get(name)
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f"Falta el secreto {name} en Colab")
    os.environ[name] = value
os.environ["MLFLOW_TRACKING_URI"] = "https://ml.pettrace.app"
os.environ["PETTRACE_EXECUTION_ID"] = uuid.uuid4().hex
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_registry_uri(os.environ["MLFLOW_TRACKING_URI"])

EXPERIMENT = "E4-TTA"
MODEL_URI = "models:/pettrace-embedder/19"
TRAINING_DATASET_VERSION = "v2"
DATASET_COMMIT = "8b12db0dce41040a821f9792bea5b352342263ce"  # benchmark reservado; no se carga
LOG_TO_MLFLOW = True
REPORT = Path("reports/e4_inference_ablation") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
REPORT.mkdir(parents=True, exist_ok=False)


In [ ]:
from huggingface_hub.utils import disable_progress_bars
from pettrace.data import training
from pettrace.evaluation.core import load_model
from pettrace.training.lora import PreparedPreprocessor

disable_progress_bars()
prepared = training(version=TRAINING_DATASET_VERSION, token=os.environ["HF_TOKEN"])
model = load_model(MODEL_URI)
if not model.golden_ok or model.registry_version != "19":
    raise RuntimeError("El modelo E4 fijado no superó la verificación")
preprocessor = PreparedPreprocessor(model.pipeline, prepared)
validation_rows = [row for row in prepared.rows if row["split"] == "val"]
if not validation_rows or {row["split"] for row in validation_rows} != {"val"}:
    raise RuntimeError("La muestra debe contener únicamente validación")
print(f"E4 verificado: {model.model_version} | validación: {len(validation_rows):,} fotos")


In [ ]:
design = {
    "objective": "Compare online-compatible inference refinements without retraining",
    "scope": "validation_only",
    "model": MODEL_URI,
    "dataset_version": TRAINING_DATASET_VERSION,
    "dataset_fingerprint": prepared.fingerprint,
    "baseline": {"representation": "single", "identity_aggregation": "max"},
    "selection": ["validation.macro_species_rank1", "validation.macro_species_mAP"],
    "protected_evaluation_commit": DATASET_COMMIT,
}
(REPORT / "experiment-design.json").write_text(json.dumps(design, indent=2) + "\n")


## Fase 2. Representación y entrenamiento

No existe entrenamiento en esta ablación. El checkpoint E4 permanece congelado. Cada recorte YOLO se representa una vez en orientación original y otra reflejada; el vector TTA es `normalize(normalize(original) + normalize(flip))`.


In [ ]:
import pandas as pd
from pettrace.evaluation.research import validation_inference_diagnostic
from pettrace.tracking import save_provenance

diagnostic = validation_inference_diagnostic(
    model.pipeline, validation_rows, preprocessor.image, batch=32)
(REPORT / "validation-inference-diagnostic.json").write_text(
    json.dumps(diagnostic, indent=2) + "\n")
save_provenance(REPORT, model)
results = pd.DataFrame(diagnostic["records"]).sort_values(
    ["validation.macro_species_rank1", "validation.macro_species_mAP"], ascending=False)
display(results.style.format({column: "{:.4f}" for column in results if column.startswith("validation.")}))


## Fase 3. Evaluación y análisis de resultados

La comparación es exclusivamente de validación. Rank-1 usa la regla de agregación indicada; mAP continúa siendo foto a foto, por lo que cambiar solo la agregación no puede alterar mAP. TTA sí puede modificar ambas métricas.


In [ ]:
baseline = next(row for row in diagnostic["records"]
                if row["representation"] == "single" and row["identity_aggregation"] == "max")
comparison = results.copy()
comparison["delta_rank1"] = comparison["validation.macro_species_rank1"] - baseline["validation.macro_species_rank1"]
comparison["delta_mAP"] = comparison["validation.macro_species_mAP"] - baseline["validation.macro_species_mAP"]
winner = comparison.iloc[0].to_dict()
analysis = {
    "baseline": baseline, "validation_winner": winner,
    "eligible_for_test_ablation": bool(winner["delta_rank1"] > 0 or
        (winner["delta_rank1"] == 0 and winner["delta_mAP"] > 0)),
    "claim_limit": "Validation evidence only; no held-out result has been computed.",
}
(REPORT / "analysis.json").write_text(json.dumps(analysis, indent=2, default=float) + "\n")
display(comparison[["representation", "identity_aggregation", "delta_rank1", "delta_mAP"]])
print("Candidata para una ablación posterior en test:", analysis["eligible_for_test_ablation"])


In [ ]:
from pettrace.tracking import publish_validation_diagnostic, source_identity

assert LOG_TO_MLFLOW is True
run_id = publish_validation_diagnostic(
    diagnostic, model, prepared, REPORT, source_identity()["source_sha256"],
    experiment=EXPERIMENT, expected_model_uri=MODEL_URI)
publication = {"published": True, "run_id": run_id}
print(f"Publicado: https://ml.pettrace.app/#/experiments/3/runs/{run_id}")


### Límites

- La selección usa una sola validación conocida y no estima incertidumbre poblacional.
- El costo duplica inferencias por consulta cuando se activa TTA.
- La agregación presupone varias fotos registradas por identidad; `top-2 mean` cae naturalmente a una foto cuando solo existe una.
- Ningún resultado de este notebook debe presentarse como mejora final hasta ejecutar una ablación fijada y separada sobre ambos tests.

Métodos externos: CLIP (Radford et al., 2021). El pooling TTA, las reglas de agregación y el criterio de selección son decisiones experimentales de esta investigación.
